<a href="https://colab.research.google.com/github/Anorwed/vkr/blob/main/picocolabqwen2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Для запуска, нажмите кнопку ниже, слвева от надписи 'Показать код'.
import os
import time
import json
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

def run_installation(token):
    bot_token = token.strip()
    if not bot_token:
        print("❌ Ошибка: Токен не может быть пустым!")
        return

    print(f"--- Начинаю процесс установки ---")

    # Подготовка системы
    print("🔧 Подготовка системы...")
    get_ipython().system('apt-get update -qq && apt-get install -y zstd pciutils net-tools')

    # Установка Ollama
    print("🦙 Установка Ollama...")
    get_ipython().system('curl -fsSL https://ollama.com/install.sh | sh')

    # Запуск Ollama
    print("🚀 Запуск Ollama сервера...")
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    get_ipython().system('nohup ollama serve > ollama.log 2>&1 &')
    time.sleep(15)

    # Проверка запуска Ollama
    print("🔍 Проверка Ollama...")
    for i in range(5):
        result = get_ipython().getoutput('curl -s http://localhost:11434/api/tags | head -c 100')
        if result and len(result) > 0 and "error" not in str(result).lower():
            print(f"✅ Ollama запущен")
            break
        time.sleep(3)

    # Загружаем модель с поддержкой tools (Qwen 2.5 вместо Gemma 3)
    print("📥 Загрузка модели Qwen 2.5 (7b) - поддерживает tools...")
    get_ipython().system('ollama pull qwen2.5:7b')

    # Проверка модели
    print("🔍 Проверка модели...")
    time.sleep(3)
    models = get_ipython().getoutput('ollama list')
    print("Доступные модели:", models)

    # Загрузка PicoClaw
    print("📥 Загрузка PicoClaw Gateway...")
    get_ipython().system('wget -q --show-progress https://github.com/sipeed/picoclaw/releases/latest/download/picoclaw_Linux_x86_64.tar.gz')
    get_ipython().system('tar -xzf picoclaw_Linux_x86_64.tar.gz')
    get_ipython().system('chmod +x picoclaw')

    # Создаем директории
    os.makedirs("/content/workspace", exist_ok=True)
    os.makedirs("/root/.picoclaw", exist_ok=True)

    # Создаем конфиг с Qwen 2.5 (поддерживает tools)
    config_path = Path.home() / ".picoclaw" / "config.json"

    config_data = {
        "agents": {
            "defaults": {
                "workspace": "/content/workspace",
                "model": "ollama/qwen2.5:7b",  # Модель с поддержкой tools
                "max_tokens": 4096,
                "temperature": 0.7,
                "max_tool_iterations": 10,
                "restrict_to_workspace": True
            }
        },
        "providers": {
            "ollama": {
                "api_key": "ollama",
                "api_base": "http://localhost:11434/v1"
            }
        },
        "channels": {
            "telegram": {
                "enabled": True,
                "token": bot_token,
                "allow_from": [],
                "polling_interval": 5,
                "webhook": {
                    "enabled": False
                }
            }
        },
        "gateway": {
            "host": "0.0.0.0",
            "port": 18790
        }
    }

    # Записываем конфиг
    with open(config_path, "w") as f:
        json.dump(config_data, f, indent=4)

    print(f"✅ Конфигурация записана")
    print(f"📋 Используемая модель: qwen2.5:7b (поддерживает tools)")

    print("\n🚀 Запускаю PicoClaw Gateway...")
    print("💡 Бот должен отвечать без ошибок 'does not support tools'\n")

    # Запуск
    get_ipython().system('./picoclaw gateway 2>&1 | tee picoclaw.log')

# --- Интерфейс ---

token_input = widgets.Password(
    value='',
    placeholder='Вставьте API токен бота от @BotFather',
    description='Bot Token:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

start_button = widgets.Button(
    description='🚀 Запустить установку',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='200px')
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        run_installation(token_input.value)

start_button.on_click(on_button_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>🦞 PicoClaw + Ollama + Telegram Bot</h3>"),
    widgets.HTML("<p>Используем Qwen 2.5 (7B) - поддерживает function calling/tools</p>"),
    token_input,
    start_button,
    output
]))
